# Task 1 — Inconsistencies and duplicates
CSEN1095 Data Engineering · W26

Goal: turn `club_signups.csv` into a table with **one row per student per club** and every value spelled one way.
The raw file is never modified; every fix below is code.

In [1]:
import re
import pandas as pd

raw = pd.read_csv("club_signups.csv")   # the file as given, never modified
cleaned = raw.copy()                    # every fix is applied to this copy

print("raw:", raw.shape, "  cleaned (start):", cleaned.shape)

raw: (39, 9)   cleaned (start): (39, 9)


## Part A — Inconsistencies
### A1. Look first

In [2]:
print("shape:", raw.shape)
print()
print(raw.dtypes)

shape: (39, 9)

signup_id       str
student_id      str
full_name       str
email           str
faculty         str
city            str
club            str
signed_up_at    str
fee_paid        str
dtype: object


In [3]:
def profile(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per column: its type, the Python types inside it, nulls, distinct values, range and top values."""
    rows = {}
    for name, column in frame.items():
        numeric = pd.api.types.is_numeric_dtype(column)
        rows[name] = {
            "dtype": str(column.dtype),
            "python types": ", ".join(sorted(column.dropna().map(lambda v: type(v).__name__).unique())),
            "missing": column.isna().sum(),
            "distinct": column.nunique(),
            "min": column.min() if numeric else None,
            "max": column.max() if numeric else None,
            "most common": column.value_counts().head(3).to_dict(),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


profile(raw)

,dtype,python types,missing,distinct,min,max,most common
signup_id,str,str,0,36,None,None,"{'S015': 2, 'S022': 2, 'S023': 2}"
student_id,str,str,0,15,None,None,"{'61-7344': 6, '55-9245': 5, '61-4844': 4}"
full_name,str,str,0,24,None,None,"{'Laila Mostafa': 4, 'Hana Zaki': 3, 'Ziad Sal..."
email,str,str,0,18,None,None,"{'ali.ibrahim@student.guc.edu.eg': 5, 'mohamed..."
faculty,str,str,0,13,None,None,"{'Pharmacy': 7, 'EMS': 7, 'IET': 5}"
city,str,str,0,9,None,None,"{'Cairo': 13, 'Alexandria': 11, 'Giza': 4}"
club,str,str,0,13,None,None,"{'Robotics': 8, 'Music': 5, 'Chess': 4}"
signed_up_at,str,str,0,36,None,None,"{'2026-09-15 13:39': 2, '2026-09-16 01:39': 2,..."
fee_paid,str,str,0,9,None,None,"{'yes': 7, '1': 7, 'false': 6}"


In [4]:
for col in ["faculty", "club", "city", "fee_paid"]:
    print(raw[col].value_counts(), end="\n\n")

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
MET                                       3
iet                                       3
Engineering and Materials Science         1
Pharma                                    1
met                                       1
Media Engineering and Technology          1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Chess Club     3
Football       3
Soccer         2
music          2
chess          2
debate club    1
Music          1
Debate         1
Name: count, dtype: int64

city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1

### A2. Faculty, club, city
Mechanical normalisation first (strip spaces, lower-case), then an explicit dictionary for what is left.
The mapping is **strict**: an unknown value raises an error instead of being silently passed through.

In [5]:
def normalise(series: pd.Series) -> pd.Series:
    """Mechanical step: trim, collapse inner whitespace, lower-case."""
    return series.str.strip().str.replace(r"\s+", " ", regex=True).str.lower()


def map_strict(series: pd.Series, aliases: dict, column: str) -> pd.Series:
    """Normalise, then map through the dictionary. Fails loudly on any value the dictionary does not know."""
    mapped = normalise(series).map(aliases)
    unknown = sorted(series[mapped.isna()].unique())
    assert not unknown, f"{column}: values missing from the dictionary: {unknown}"
    return mapped

In [6]:
FACULTIES = {"MET", "IET", "EMS", "Pharmacy", "Management"}
CLUBS = {"Robotics", "Debate", "Music", "Football", "Chess"}
CITIES = {"Cairo", "Giza", "Alexandria"}

# keys are already normalised (lower-case, trimmed)
FACULTY_ALIASES = {
    "met": "MET", "media engineering and technology": "MET",
    "iet": "IET", "information engineering and technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy", "pharma": "Pharmacy",
    "management": "Management",
}
CLUB_ALIASES = {
    "robotics": "Robotics",
    "debate": "Debate", "debating": "Debate", "debate club": "Debate",
    "music": "Music",
    "football": "Football", "soccer": "Football",
    "chess": "Chess", "chess club": "Chess",
}
CITY_ALIASES = {
    "cairo": "Cairo",
    "giza": "Giza", "el giza": "Giza",
    "alexandria": "Alexandria", "alex": "Alexandria",
}

cleaned["faculty"] = map_strict(raw["faculty"], FACULTY_ALIASES, "faculty")
cleaned["club"] = map_strict(raw["club"], CLUB_ALIASES, "club")
cleaned["city"] = map_strict(raw["city"], CITY_ALIASES, "city")

In [7]:
# ✓ check: only the canonical values remain
assert set(cleaned["faculty"]) == FACULTIES, set(cleaned["faculty"])
assert set(cleaned["club"]) == CLUBS, set(cleaned["club"])
assert set(cleaned["city"]) == CITIES, set(cleaned["city"])

for col in ["faculty", "club", "city"]:
    print("✓", col, cleaned[col].value_counts().to_dict())

✓ faculty {'IET': 12, 'Pharmacy': 9, 'EMS': 9, 'MET': 5, 'Management': 4}
✓ club {'Football': 9, 'Chess': 9, 'Music': 8, 'Robotics': 8, 'Debate': 5}
✓ city {'Alexandria': 16, 'Cairo': 16, 'Giza': 7}


### A3. Names and emails
Names: collapse every run of whitespace to one space, trim, and use title case, so `"  Mohamed  Adel "` and `"MOHAMED ADEL"` become `Mohamed Adel`.
Emails: trim and lower-case.

In [8]:
cleaned["full_name"] = (
    raw["full_name"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
)
cleaned["email"] = raw["email"].str.strip().str.lower()

# ✓ checks
assert not cleaned["full_name"].str.contains(r"\s{2,}|^\s|\s$", regex=True).any()
assert (cleaned["full_name"] == cleaned["full_name"].str.title()).all()
assert (cleaned["email"] == cleaned["email"].str.strip().str.lower()).all()
assert cleaned["student_id"].str.fullmatch(r"\d{2}-\d{4}").all()   # ids are clean already

print("raw spellings of the names :", raw["full_name"].nunique())
print("cleaned spellings          :", cleaned["full_name"].nunique())
print(sorted(cleaned["full_name"].unique()))

raw spellings of the names : 24
cleaned spellings          : 14
['Ahmed Adel', 'Ahmed Zaki', 'Ali Ibrahim', 'Farida Ibrahim', 'Hana Zaki', 'Karim Kamel', 'Laila Mostafa', 'Mariam Kamel', 'Mohamed Adel', 'Omar Adel', 'Salma Ibrahim', 'Sara Zaki', 'Youssef Fathy', 'Ziad Saleh']


### A4. `fee_paid` → boolean

In [9]:
BOOLEAN_ALIASES = {
    "yes": True, "y": True, "true": True, "1": True,
    "no": False, "n": False, "false": False, "0": False,
}

fee = raw["fee_paid"].astype(str).str.strip().str.lower().map(BOOLEAN_ALIASES)
assert fee.notna().all(), raw.loc[fee.isna(), "fee_paid"].unique()
cleaned["fee_paid"] = fee.astype(bool)

# ✓ check
assert cleaned["fee_paid"].dtype == bool
print("✓", cleaned["fee_paid"].dtype, cleaned["fee_paid"].value_counts().to_dict())

✓ bool {True: 27, False: 12}


### A5. `signed_up_at`
Two formats are mixed in the column. First find them.

In [10]:
ISO = r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}"
SLASH = r"\d{2}/\d{2}/\d{4} \d{2}:\d{2}"

stamp = raw["signed_up_at"].str.strip()
is_iso = stamp.str.fullmatch(ISO)
is_slash = stamp.str.fullmatch(SLASH)
assert (is_iso | is_slash).all(), stamp[~(is_iso | is_slash)].tolist()   # nothing outside the two formats

print("ISO   (YYYY-MM-DD HH:MM):", is_iso.sum(), "rows, e.g.", stamp[is_iso].iloc[0])
print("slash (??/??/YYYY HH:MM):", is_slash.sum(), "rows, e.g.", stamp[is_slash].iloc[0])

ISO   (YYYY-MM-DD HH:MM): 25 rows, e.g. 2026-09-15 13:39
slash (??/??/YYYY HH:MM): 14 rows, e.g. 15/09/2026 03:12


**Which part of the slash format is the day?** Look at the two numbers before the year: if the first one goes above 12, it cannot be a month.

In [11]:
parts = stamp[is_slash].str.extract(r"(?P<first>\d{2})/(?P<second>\d{2})/").astype(int)
print("first number  range:", parts["first"].min(), "-", parts["first"].max())
print("second number range:", parts["second"].min(), "-", parts["second"].max())

assert parts["first"].max() > 12     # so the first number must be the DAY …
assert parts["second"].max() <= 12   # … and the second number the MONTH  →  dd/mm/YYYY

first number  range: 13 - 18
second number range: 9 - 9


In [12]:
parsed = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")
parsed[is_iso] = pd.to_datetime(stamp[is_iso], format="%Y-%m-%d %H:%M")
parsed[is_slash] = pd.to_datetime(stamp[is_slash], format="%d/%m/%Y %H:%M")   # day first
cleaned["signed_up_at"] = parsed

# ✓ checks
assert cleaned["signed_up_at"].notna().all()
assert pd.api.types.is_datetime64_any_dtype(cleaned["signed_up_at"])
# everything was submitted during the one-week form window (13–18 Sep 2026); a swapped day/month would break this
assert cleaned["signed_up_at"].between("2026-09-13", "2026-09-19").all()
print("✓", cleaned["signed_up_at"].min(), "→", cleaned["signed_up_at"].max())

✓ 2026-09-13 14:36:00 → 2026-09-18 23:36:00


In [13]:
cleaned.dtypes

signup_id                  str
student_id                 str
full_name                  str
email                      str
faculty                    str
city                       str
club                       str
signed_up_at    datetime64[ns]
fee_paid                  bool
dtype: object

## Part B — Duplicates
### B1. Exact duplicates (the same form submission twice)

In [14]:
start_rows = len(cleaned)

exact_mask = cleaned.duplicated(keep="first")
n_exact = int(exact_mask.sum())
print("rows at the start of Part B   :", start_rows)
print("exact duplicate rows (extra copies):", n_exact)
print("same count on the untouched raw file:", int(raw.duplicated().sum()))
print()
cleaned[cleaned.duplicated(keep=False)].sort_values("signup_id")

rows at the start of Part B   : 39
exact duplicate rows (extra copies): 3
same count on the untouched raw file: 3



,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False


In [15]:
cleaned = cleaned.drop_duplicates(keep="first")
after_exact = len(cleaned)
print(f"{start_rows} → {after_exact}  ({start_rows - after_exact} removed)")

assert cleaned["signup_id"].is_unique   # each form submission now appears once

39 → 36  (3 removed)


### B2. The same sign-up submitted again
The same `student_id` and `club` under different `signup_id`s. Keep the **latest** submission, ordered by the parsed
`signed_up_at` (not by the text, and not by `signup_id`).

In [16]:
key = ["student_id", "club"]
repeats = cleaned[cleaned.duplicated(key, keep=False)].sort_values(key + ["signed_up_at"])
repeats[["signup_id", "student_id", "full_name", "club", "signed_up_at", "fee_paid"]]

,signup_id,student_id,full_name,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,Debate,2026-09-17 11:15:00,True
35,S017,61-7344,Laila Mostafa,Debate,2026-09-15 18:01:00,False
25,S034,61-7344,Laila Mostafa,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,Football,2026-09-17 23:36:00,False
38,S036,61-7344,Laila Mostafa,Football,2026-09-18 23:36:00,True


In [17]:
# no two submissions of the same student+club share a timestamp, so "latest" is unambiguous
assert not repeats.duplicated(key + ["signed_up_at"]).any()

cleaned = (
    cleaned.sort_values("signed_up_at")
           .drop_duplicates(subset=key, keep="last")
           .sort_values("signup_id")
           .reset_index(drop=True)
)
after_resubmit = len(cleaned)
print(f"{after_exact} → {after_resubmit}  ({after_exact - after_resubmit} removed)")

36 → 32  (4 removed)


### B3. Check: one row per student per club

In [18]:
assert not cleaned.duplicated(["student_id", "club"]).any()
print("✓ student_id + club is unique:", len(cleaned), "rows")

✓ student_id + club is unique: 32 rows


### Final checks and save

In [19]:
assert set(cleaned["faculty"]) <= FACULTIES
assert set(cleaned["club"]) <= CLUBS
assert set(cleaned["city"]) <= CITIES
assert cleaned["fee_paid"].dtype == bool
assert pd.api.types.is_datetime64_any_dtype(cleaned["signed_up_at"])

cleaned.to_csv("club_signups_clean.csv", index=False)

check = pd.read_csv("club_signups_clean.csv")
print("saved:", check.shape)
cleaned

saved: (32, 9)


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
5,S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
6,S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
7,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
8,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
9,S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False


## Part C — Numbers for the README
The next cells produce the figures quoted in the README paragraph.

In [20]:
print(f"started with            : {len(raw)}")
print(f"exact duplicates removed: {n_exact}")
print(f"re-submissions removed  : {after_exact - after_resubmit}")
print(f"rows left               : {len(cleaned)}")

started with            : 39
exact duplicates removed: 3
re-submissions removed  : 4
rows left               : 32


**Does the order of steps matter?** Run the "same student + club" step on the raw table, *before* the club spellings are fixed, and compare.

In [21]:
# Same two steps, but with the club column still in its raw spelling (only whitespace trimmed, so the experiment is fair)
wrong = raw.assign(signed_up_at=parsed, club=raw["club"].str.strip()).drop_duplicates()
n_wrong_exact = len(raw) - len(wrong)

wrong_after = (
    wrong.sort_values("signed_up_at").drop_duplicates(subset=["student_id", "club"], keep="last")
)
n_wrong_resubmit = len(wrong) - len(wrong_after)

print(f"unfixed spellings: exact removed = {n_wrong_exact}, re-submissions removed = {n_wrong_resubmit}, rows left = {len(wrong_after)}")
print(f"fixed spellings  : exact removed = {n_exact}, re-submissions removed = {after_exact - after_resubmit}, rows left = {len(cleaned)}")
print()
print("distinct club spellings in raw:", raw["club"].str.strip().nunique(), "-> canonical:", cleaned["club"].nunique())
print("student_id + club pairs still duplicated in the 'wrong order' result:",
      int(wrong_after.assign(club=map_strict(wrong_after["club"], CLUB_ALIASES, "club")).duplicated(["student_id", "club"]).sum()))

unfixed spellings: exact removed = 3, re-submissions removed = 2, rows left = 34
fixed spellings  : exact removed = 3, re-submissions removed = 4, rows left = 32

distinct club spellings in raw: 12 -> canonical: 5
student_id + club pairs still duplicated in the 'wrong order' result: 2


**Two different students, one name.** Dedup uses `student_id`, never the name.

In [22]:
ids_per_name = cleaned.groupby("full_name")["student_id"].nunique()
shared = ids_per_name[ids_per_name > 1]
print("names shared by more than one student_id:")
print(shared)
print()
print(cleaned[cleaned["full_name"].isin(shared.index)]
      .sort_values(["full_name", "student_id"])[["student_id", "full_name", "faculty", "club", "signed_up_at"]])

# if we had (wrongly) deduplicated on name + club, how many real sign-ups would have been destroyed?
by_name = cleaned.drop_duplicates(["full_name", "club"])
print(f"\nname+club would keep {len(by_name)} rows instead of {len(cleaned)} → {len(cleaned) - len(by_name)} real sign-ups lost")

names shared by more than one student_id:
full_name
Mohamed Adel    2
Name: student_id, dtype: int64

   student_id     full_name faculty      club        signed_up_at
1     55-2992  Mohamed Adel     IET    Debate 2026-09-13 15:38:00
20    55-2992  Mohamed Adel     IET  Robotics 2026-09-16 16:16:00
7     61-4844  Mohamed Adel     EMS     Music 2026-09-14 23:32:00
8     61-4844  Mohamed Adel     EMS  Robotics 2026-09-15 01:05:00
30    61-4844  Mohamed Adel     EMS    Debate 2026-09-17 11:15:00



name+club would keep 30 rows instead of 32 → 2 real sign-ups lost
